## NeuroScan MRI - PyTorch Windows Edition

This notebook contains the MRI-only pipeline for early Alzheimer's classification.

Target outputs:
- `saved_models/`
- `plots/`
- `reports/`
- executed notebook snapshot

Metrics retained where practical:
- `final_train_loss`, `final_train_acc`, `final_val_loss`, `final_val_acc`
- `test_loss`, `test_acc`
- confusion matrices, ROC curves, classification reports
- MRI risk summary

In [1]:
# Section 2: Set Up PyTorch Dependencies and Reproducibility
import json
import math
import os
import random
import subprocess
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (auc,classification_report,confusion_matrix,f1_score,precision_score,recall_score,roc_curve,)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset
from torchvision import datasets,models,transforms
from torchvision.io import read_image
warnings.filterwarnings("ignore")
os.environ.setdefault("CUDA_VISIBLE_DEVICES","0")
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
CUDA_AVAILABLE=torch.cuda.is_available()
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.set_device(0)
    torch.backends.cudnn.benchmark=True
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
else:
    torch.backends.cudnn.benchmark=False
def _find_project_root()->Path:
    env_root=os.environ.get("NS_PROJECT_ROOT")
    if env_root and (Path(env_root)/"MRI").exists():
        return Path(env_root)
    for candidate in [Path.cwd(),*Path.cwd().parents]:
        if (candidate/"MRI").exists():
            return candidate
    return Path.cwd()
PROJECT_ROOT=_find_project_root()
MRI_TRAIN_DIR=PROJECT_ROOT/"MRI"/"train"
MRI_TEST_DIR=PROJECT_ROOT/"MRI"/"test"
SAVED_MODELS_DIR=PROJECT_ROOT/"saved_models"
REPORTS_DIR=PROJECT_ROOT/"reports"
PLOTS_DIR=PROJECT_ROOT/"plots"
for directory in [SAVED_MODELS_DIR,REPORTS_DIR,PLOTS_DIR]:
    directory.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device("cuda:0" if CUDA_AVAILABLE else "cpu")
if CUDA_AVAILABLE:
    gpu_name=torch.cuda.get_device_name(0)
    print(f"PyTorch {torch.__version__} | CUDA: True | GPU: {gpu_name}")
else:
    print(f"PyTorch {torch.__version__} | CUDA: False | running on CPU")
print(f"Using device: {DEVICE}")
FAST_RUN=os.environ.get("FAST_RUN","0")=="1"
FAST_EPOCHS=1
MRI_STAGE1_EPOCHS=24
MRI_STAGE2_EPOCHS=12
BATCH_SIZE=16
IMG_SIZE=224
NUM_WORKERS=0 if os.name=="nt" else max(0,min(4,os.cpu_count() or 0))
PIN_MEMORY=DEVICE.type=="cuda"
MRI_TTA_PASSES=12
MRI_STAGE1_LR=2e-4
MRI_STAGE2_LR=2e-5
MRI_WARMUP_EPOCHS=3
MRI_MODEL_PATH=SAVED_MODELS_DIR/"best_mri_model.pth"
MRI_META_PATH=SAVED_MODELS_DIR/"mri_metadata.json"
SUMMARY_JSON=REPORTS_DIR/"neuroscan_metrics.json"
SUMMARY_CSV=REPORTS_DIR/"neuroscan_metrics.csv"
REPORT_JSON=REPORTS_DIR/"neuroscan_report.json"
print(f"Project root: {PROJECT_ROOT}")
print(f"MRI train/test: {MRI_TRAIN_DIR} | {MRI_TEST_DIR}")
print(f"Batch size: {BATCH_SIZE} | Workers: {NUM_WORKERS} | FAST_RUN={FAST_RUN}")
def _gpu_stats_line()->str:
    if DEVICE.type!="cuda":
        return "cpu-mode"
    try:
        out=subprocess.check_output(["nvidia-smi","--query-gpu=utilization.gpu,memory.used,memory.total","--format=csv,noheader,nounits"],text=True,timeout=2,).strip()
        return out
    except Exception as exc:
        return f"unavailable ({exc})"
class GpuEpochLogger:
    def on_epoch_begin(self,epoch:int):
        print(f"[GPU][epoch {epoch+1} begin] {_gpu_stats_line()}")
    def on_epoch_end(self,epoch:int,logs:dict):
        logs=logs or {}
        print(f"[GPU][epoch {epoch+1} end] {_gpu_stats_line()} | loss={logs.get('loss',float('nan')):.4f} | val_loss={logs.get('val_loss',float('nan')):.4f} | acc={logs.get('accuracy',float('nan')):.4f} | val_acc={logs.get('val_accuracy',float('nan')):.4f}")
@dataclass
class MetricsSummary:
    modality:str
    final_train_loss:float
    final_train_acc:float
    final_val_loss:float
    final_val_acc:float
    test_loss:float
    test_acc:float
    precision:float
    recall:float
    f1_score:float
    confusion_matrix:list
    report:dict
class WarmupCosineLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self,optimizer,total_epochs,warmup_epochs=3,min_lr=1e-6,last_epoch=-1):
        self.total_epochs=max(1,int(total_epochs))
        self.warmup_epochs=max(1,int(warmup_epochs))
        self.min_lr=float(min_lr)
        super().__init__(optimizer,last_epoch)
    def get_lr(self):
        step=max(0,self.last_epoch)
        if step<self.warmup_epochs:
            scale=float(step+1)/float(self.warmup_epochs)
            return [base_lr*scale for base_lr in self.base_lrs]
        progress=(step-self.warmup_epochs)/max(1,self.total_epochs-self.warmup_epochs)
        cosine=0.5*(1+math.cos(math.pi*min(1.0,progress)))
        return [self.min_lr+(base_lr-self.min_lr)*cosine for base_lr in self.base_lrs]
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,label_smoothing=0.0):
        super().__init__()
        self.gamma=gamma
        self.weight=weight
        self.label_smoothing=label_smoothing
    def forward(self,logits,targets):
        if targets.ndim==2:
            targets=targets.argmax(dim=1)
        ce=F.cross_entropy(logits,targets,weight=self.weight,reduction="none",label_smoothing=self.label_smoothing)
        pt=torch.exp(-ce)
        loss=((1-pt)**self.gamma)*ce
        return loss.mean()
def save_json(path:Path,payload:dict):
    path.write_text(json.dumps(payload,indent=2),encoding="utf-8")
print("PyTorch setup complete.")

PyTorch 2.11.0+cpu | CUDA: False | running on CPU
Using device: cpu
Project root: c:\Users\Surya Thejas\OneDrive\Desktop\Early_Alzheimers
MRI train/test: c:\Users\Surya Thejas\OneDrive\Desktop\Early_Alzheimers\MRI\train | c:\Users\Surya Thejas\OneDrive\Desktop\Early_Alzheimers\MRI\test
Batch size: 16 | Workers: 0 | FAST_RUN=False
PyTorch setup complete.


In [2]:
# Section 3: Load Data and Build PyTorch Dataset/DataLoader
from collections import Counter

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

mri_train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
mri_eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def stratified_split_indices(targets, val_ratio=0.2):
    targets = np.asarray(targets)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=SEED)
    train_idx, val_idx = next(splitter.split(np.zeros(len(targets)), targets))
    return train_idx.tolist(), val_idx.tolist()


def build_modal_dataloaders(train_dir: Path, test_dir: Path, train_tfms, eval_tfms, batch_size=BATCH_SIZE):
    base_train = datasets.ImageFolder(train_dir, transform=None)
    base_test = datasets.ImageFolder(test_dir, transform=None)
    class_names = base_train.classes
    targets = [y for _, y in base_train.samples]
    train_idx, val_idx = stratified_split_indices(targets, val_ratio=0.2)

    class IndexedImageFolderDataset(Dataset):
        def __init__(self, folder_dataset, indices, transform=None):
            self.folder_dataset = folder_dataset
            self.indices = list(indices)
            self.transform = transform
            self.targets = [folder_dataset.samples[i][1] for i in self.indices]

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, idx):
            sample_idx = self.indices[idx]
            path, label = self.folder_dataset.samples[sample_idx]
            image = read_image(path)
            if image.shape[0] == 1:
                image = image.repeat(3, 1, 1)
            if self.transform is not None:
                image = self.transform(image)
            return image, label

    train_ds = IndexedImageFolderDataset(base_train, train_idx, transform=train_tfms)
    val_ds = IndexedImageFolderDataset(base_train, val_idx, transform=eval_tfms)
    test_ds = IndexedImageFolderDataset(base_test, list(range(len(base_test))), transform=eval_tfms)

    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    class_counts = Counter(train_ds.targets)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(sorted(class_counts.keys())),
        y=np.array(train_ds.targets),
    )
    class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    return {
        "base_train": base_train,
        "base_test": base_test,
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "class_names": class_names,
        "class_weights": class_weights,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "loader_kwargs": loader_kwargs,
    }


mri_bundle = build_modal_dataloaders(MRI_TRAIN_DIR, MRI_TEST_DIR, mri_train_tfms, mri_eval_tfms)

MRI_CLASSES = mri_bundle["class_names"]
NUM_MRI_CLASSES = len(MRI_CLASSES)
MRI_CW = mri_bundle["class_weights"]

# Preserve winner metadata if it already exists so dashboard keeps using the selected best model.
existing_meta = {}
if MRI_META_PATH.exists():
    try:
        existing_meta = json.loads(MRI_META_PATH.read_text(encoding="utf-8"))
    except Exception:
        existing_meta = {}

meta_payload = {"class_names": MRI_CLASSES}
existing_kind = existing_meta.get("model_kind")
if existing_kind in {"resnet50", "resnet50_quantum_attention"}:
    meta_payload["model_kind"] = existing_kind
save_json(MRI_META_PATH, meta_payload)

print(f"MRI classes ({NUM_MRI_CLASSES}): {MRI_CLASSES}")
print(f"MRI train/val/test batches: {len(mri_bundle['train_loader'])}/{len(mri_bundle['val_loader'])}/{len(mri_bundle['test_loader'])}")
print("Class weight tensors prepared on device.")


MRI classes (4): ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']
MRI train/val/test batches: 512/128/80
Class weight tensors prepared on device.


In [3]:
# Section 3B: MRI Risk Weights for downstream scoring

def infer_mri_risk_weights(class_names):
    weights = []
    for name in class_names:
        key = name.lower()
        if "no" in key:
            weights.append(0.0)
        elif "very mild" in key:
            weights.append(0.25)
        elif "mild" in key:
            weights.append(0.6)
        elif "moderate" in key:
            weights.append(1.0)
        else:
            weights.append(0.5)
    return np.array(weights, dtype=np.float32)


MRI_RISK_WEIGHTS = infer_mri_risk_weights(MRI_CLASSES)
healthy_idx_check = int(np.argmin(MRI_RISK_WEIGHTS))
print(f"MRI risk weights ready. Healthy baseline: {MRI_CLASSES[healthy_idx_check]} (index {healthy_idx_check})")

MRI risk weights ready. Healthy baseline: No Impairment (index 2)


In [4]:
# Section 4: Convert the Model to torch.nn.Module
from torchvision.models import EfficientNet_B3_Weights, ResNet50_Weights


def _load_backbone(backbone_name: str):
    if backbone_name == "resnet50":
        try:
            return models.resnet50(weights=ResNet50_Weights.DEFAULT), "pretrained"
        except Exception:
            return models.resnet50(weights=None), "random"
    if backbone_name == "efficientnet_b3":
        try:
            return models.efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT), "pretrained"
        except Exception:
            return models.efficientnet_b3(weights=None), "random"
    raise ValueError(f"Unsupported backbone: {backbone_name}")


class QuantumAttentionBlock(nn.Module):
    """Quantum-inspired attention block using phase modulation + channel gating."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.phase = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=True),
        )
        self.gate = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x):
        pooled = self.pool(x)
        phase = torch.tanh(self.phase(pooled)) * math.pi
        amplitude = torch.cos(phase)
        channel_gate = self.gate(pooled)
        return x * (0.5 + 0.5 * amplitude) * channel_gate


class QuantumAttentionResNet50(nn.Module):
    def __init__(self, num_classes: int, dropout1=0.45, dropout2=0.30):
        super().__init__()
        backbone, init_mode = _load_backbone("resnet50")
        self.init_mode = init_mode
        self.backbone_name = "resnet50_quantum_attention"

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )
        self.q_attention = QuantumAttentionBlock(channels=2048, reduction=16)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout1),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feats = self.stem(x)
        feats = self.q_attention(feats)
        feats = self.pool(feats)
        feats = torch.flatten(feats, 1)
        return self.head(feats)


class BackboneClassifier(nn.Module):
    def __init__(self, backbone_name: str, num_classes: int, dropout1=0.5, dropout2=0.35):
        super().__init__()
        self.backbone_name = backbone_name
        backbone, init_mode = _load_backbone(backbone_name)
        self.init_mode = init_mode
        if backbone_name == "resnet50":
            in_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.backbone = backbone
            hidden1, hidden2 = 512, 256
        elif backbone_name == "efficientnet_b3":
            in_features = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity()
            self.backbone = backbone
            hidden1, hidden2 = 512, 256
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.head = nn.Sequential(
            nn.Linear(in_features, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout1),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout2),
            nn.Linear(hidden2, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)
        if feats.ndim > 2:
            feats = torch.flatten(feats, 1)
        return self.head(feats)


def freeze_backbone(model: nn.Module):
    for name, param in model.named_parameters():
        if "head" not in name and "q_attention" not in name:
            param.requires_grad = False


def unfreeze_last_blocks(model: nn.Module, backbone_name: str, n_blocks: int):
    _ = n_blocks
    for _, param in model.named_parameters():
        param.requires_grad = False

    if backbone_name == "resnet50":
        for name, param in model.backbone.named_parameters():
            if any(block in name for block in ["layer4", "layer3"]):
                param.requires_grad = True
    elif backbone_name == "resnet50_quantum_attention":
        for name, param in model.stem.named_parameters():
            if any(block in name for block in ["7", "6"]):
                param.requires_grad = True
        for _, param in model.q_attention.named_parameters():
            param.requires_grad = True
    elif backbone_name == "efficientnet_b3":
        for name, param in model.backbone.named_parameters():
            if any(block in name for block in ["features.6", "features.7"]):
                param.requires_grad = True

    for name, param in model.named_parameters():
        if "head" in name:
            param.requires_grad = True


def build_model(backbone_name: str, num_classes: int):
    if backbone_name == "resnet50_quantum_attention":
        model = QuantumAttentionResNet50(num_classes)
    else:
        model = BackboneClassifier(backbone_name, num_classes)
    return model.to(DEVICE)


def count_params(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


mri_model = build_model("resnet50", NUM_MRI_CLASSES)
freeze_backbone(mri_model)

total, trainable = count_params(mri_model)
print(f"MRI model: total params={total:,} | trainable={trainable:,} | init={mri_model.init_mode}")
print("Available backbones: resnet50, resnet50_quantum_attention")
print("PyTorch model definitions ready.")

MRI model: total params=24,691,012 | trainable=1,182,980 | init=pretrained
Available backbones: resnet50, resnet50_quantum_attention
PyTorch model definitions ready.


In [5]:
# Section 5: Implement Training, Validation, and Metric Logging

def accuracy_from_logits(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def batch_to_device(batch):
    x, y = batch
    return x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)


def train_one_epoch(model, loader, criterion, optimizer, scaler, mixup_alpha=0.0):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_count = 0
    for images, targets in loader:
        images, targets = batch_to_device((images, targets))

        if mixup_alpha > 0 and np.random.rand() > 0.5:
            batch_size = images.size(0)
            index = torch.randperm(batch_size, device=images.device)
            lam = float(np.random.beta(mixup_alpha, mixup_alpha))
            images = lam * images + (1 - lam) * images[index, :]
            targets_a, targets_b = targets, targets[index]
        else:
            targets_a = targets_b = targets
            lam = 1.0

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(images)
            loss_a = criterion(logits, targets_a)
            if mixup_alpha > 0:
                loss_b = criterion(logits, targets_b)
                loss = lam * loss_a + (1 - lam) * loss_b
            else:
                loss = loss_a

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * targets_a.size(0)
        running_correct += (logits.argmax(dim=1) == targets_a).sum().item()
        running_count += targets_a.size(0)
    return running_loss / max(1, running_count), running_correct / max(1, running_count)


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_count = 0
    all_targets = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for images, targets in loader:
            images, targets = batch_to_device((images, targets))
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            probs = torch.softmax(logits, dim=1)
            running_loss += loss.item() * targets.size(0)
            running_correct += (logits.argmax(dim=1) == targets).sum().item()
            running_count += targets.size(0)
            all_targets.append(targets.detach().cpu().numpy())
            all_preds.append(logits.argmax(dim=1).detach().cpu().numpy())
            all_probs.append(probs.detach().cpu().numpy())
    y_true = np.concatenate(all_targets) if all_targets else np.array([])
    y_pred = np.concatenate(all_preds) if all_preds else np.array([])
    y_prob = np.concatenate(all_probs) if all_probs else np.array([])
    return running_loss / max(1, running_count), running_correct / max(1, running_count), y_true, y_pred, y_prob


def fit_model(
    model,
    train_loader,
    val_loader,
    class_weights,
    num_epochs,
    peak_lr,
    warmup_epochs,
    checkpoint_path: Path,
    label_names,
    stage_name: str,
    finetune=False,
    optimizer_weight_decay=1e-4,
    label_smoothing=None,
    patience_override=None,
    mixup_alpha=0.0,
 ):
    if finetune:
        unfreeze_last_blocks(model, model.backbone_name, 2)
    else:
        freeze_backbone(model)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=peak_lr,
        weight_decay=optimizer_weight_decay,
    )
    scheduler = WarmupCosineLR(
        optimizer,
        total_epochs=num_epochs,
        warmup_epochs=warmup_epochs,
        min_lr=peak_lr * 0.01,
    )

    if label_smoothing is None:
        label_smoothing = 0.08 if len(label_names) > 4 else 0.12
    criterion = FocalLoss(gamma=2.0, weight=class_weights, label_smoothing=label_smoothing)

    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    logger = GpuEpochLogger()
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_acc = -1.0
    best_state = None
    patience = int(patience_override) if patience_override is not None else (8 if finetune else 12)
    patience_counter = 0

    print(
        f"â–¶ {stage_name}: {num_epochs} epochs | finetune={finetune} | "
        f"peak_lr={peak_lr} | wd={optimizer_weight_decay} | ls={label_smoothing} | mixup={mixup_alpha}"
    )
    for epoch in range(num_epochs):
        logger.on_epoch_begin(epoch)
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, mixup_alpha)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        logger.on_epoch_end(epoch, {"loss": train_loss, "accuracy": train_acc, "val_loss": val_loss, "val_accuracy": val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save({"model_state_dict": best_state, "label_names": label_names}, checkpoint_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_train_loss = history["loss"][-1]
    final_train_acc = history["accuracy"][-1]
    final_val_loss = history["val_loss"][-1]
    final_val_acc = history["val_accuracy"][-1]
    return model, history, criterion, dict(
        final_train_loss=final_train_loss,
        final_train_acc=final_train_acc,
        final_val_loss=final_val_loss,
        final_val_acc=final_val_acc,
        best_val_acc=best_val_acc,
    )


def merge_histories(history_a, history_b):
    merged = {}
    for key in set(history_a) | set(history_b):
        merged[key] = history_a.get(key, []) + history_b.get(key, [])
    return merged


def plot_history(history, title_prefix, save_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(history["accuracy"], label="Train", color="#4f8ef7", linewidth=2)
    axes[0].plot(history["val_accuracy"], label="Validation", color="#f5a623", linewidth=2)
    axes[0].set_title(f"{title_prefix} Accuracy", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(history["loss"], label="Train", color="#4f8ef7", linewidth=2)
    axes[1].plot(history["val_loss"], label="Validation", color="#f5a623", linewidth=2)
    axes[1].set_title(f"{title_prefix} Loss", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close(fig)

print("Training helpers ready.")


Training helpers ready.


In [5]:
# Section 5A: Train and Evaluate ResNet50

def _compute_extended_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)
    if y_true.size == 0:
        raise ValueError("Empty evaluation set.")

    cm_local = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_MRI_CLASSES))
    precision_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall_w = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    confidence = float(np.mean(np.max(y_prob, axis=1)))
    class_specificity = []
    for class_index in range(NUM_MRI_CLASSES):
        tp = cm_local[class_index, class_index]
        fp = cm_local[:, class_index].sum() - tp
        fn = cm_local[class_index, :].sum() - tp
        tn = cm_local.sum() - tp - fp - fn
        class_specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    specificity = float(np.mean(class_specificity))
    y_onehot = np.eye(NUM_MRI_CLASSES, dtype=np.float32)[y_true]
    brier = float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))

    probs_max = np.max(y_prob, axis=1)
    correctness = (y_pred == y_true).astype(np.float32)
    bin_edges = np.linspace(0.0, 1.0, 11)
    cal_x = []
    cal_y = []
    ece = 0.0
    total = float(len(probs_max))
    for left, right in zip(bin_edges[:-1], bin_edges[1:]):
        if right < 1.0:
            mask = (probs_max >= left) & (probs_max < right)
        else:
            mask = (probs_max >= left) & (probs_max <= right)
        if not np.any(mask):
            continue
        mean_conf = float(np.mean(probs_max[mask]))
        frac_correct = float(np.mean(correctness[mask]))
        cal_x.append(mean_conf)
        cal_y.append(frac_correct)
        ece += abs(mean_conf - frac_correct) * (float(np.sum(mask)) / total)

    return {
        "confidence": confidence,
        "sensitivity": float(recall_w),
        "specificity": specificity,
        "f1_score": float(f1_w),
        "f1_macro": float(f1_m),
        "brier_score": brier,
        "brier_calibration": float(ece),
        "precision": float(precision_w),
        "recall": float(recall_w),
        "cal_x": cal_x,
        "cal_y": cal_y,
    }


def _run_two_stage_experiment(
    model_key,
    backbone_name,
    stage1_epochs,
    stage2_epochs,
    stage1_lr,
    stage2_lr,
    mixup_alpha,
    stage1_weight_decay,
    stage2_weight_decay,
    stage1_patience,
    stage2_patience,
    stage1_label_smoothing,
    stage2_label_smoothing,
    stage1_warmup,
    stage2_warmup,
 ):
    print(f"\n{'=' * 90}")
    print(f"Starting {model_key} experiment")
    print(f"{'=' * 90}")

    model = build_model(backbone_name, NUM_MRI_CLASSES)
    freeze_backbone(model)
    total_params, trainable_params = count_params(model)
    print(f"Model params: total={total_params:,} | trainable={trainable_params:,}")

    stage1_ckpt = SAVED_MODELS_DIR / f"{model_key}_stage1.pth"
    stage2_ckpt = SAVED_MODELS_DIR / f"{model_key}_best.pth"

    model, history1, _, stage1_metrics = fit_model(
        model,
        mri_bundle["train_loader"],
        mri_bundle["val_loader"],
        MRI_CW,
        num_epochs=stage1_epochs,
        peak_lr=stage1_lr,
        warmup_epochs=stage1_warmup,
        checkpoint_path=stage1_ckpt,
        label_names=MRI_CLASSES,
        stage_name=f"{model_key} stage 1",
        finetune=False,
        optimizer_weight_decay=stage1_weight_decay,
        label_smoothing=stage1_label_smoothing,
        patience_override=stage1_patience,
        mixup_alpha=mixup_alpha,
    )

    model, history2, criterion2, stage2_metrics = fit_model(
        model,
        mri_bundle["train_loader"],
        mri_bundle["val_loader"],
        MRI_CW,
        num_epochs=stage2_epochs,
        peak_lr=stage2_lr,
        warmup_epochs=stage2_warmup,
        checkpoint_path=stage2_ckpt,
        label_names=MRI_CLASSES,
        stage_name=f"{model_key} stage 2",
        finetune=True,
        optimizer_weight_decay=stage2_weight_decay,
        label_smoothing=stage2_label_smoothing,
        patience_override=stage2_patience,
        mixup_alpha=mixup_alpha,
    )

    combined_history = merge_histories(history1, history2)
    plot_history(history1, f"{model_key} Stage 1", PLOTS_DIR / f"{model_key}_stage1_training_curves.png")
    plot_history(history2, f"{model_key} Stage 2", PLOTS_DIR / f"{model_key}_stage2_training_curves.png")
    plot_history(combined_history, f"{model_key} Combined", PLOTS_DIR / f"{model_key}_training_curves.png")

    test_loss, test_acc, y_true, y_pred, y_prob = evaluate(model, mri_bundle["test_loader"], criterion2)
    ext = _compute_extended_metrics(y_true, y_pred, y_prob)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_MRI_CLASSES))
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=MRI_CLASSES,
        output_dict=True,
        zero_division=0,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        target_names=MRI_CLASSES,
        digits=4,
        zero_division=0,
    )

    payload = {
        "model": model,
        "model_kind": backbone_name,
        "criterion": criterion2,
        "history1": history1,
        "history2": history2,
        "history": combined_history,
        "stage1_metrics": stage1_metrics,
        "stage2_metrics": stage2_metrics,
        "final_train_loss": combined_history["loss"][-1],
        "final_train_acc": combined_history["accuracy"][-1],
        "final_val_loss": combined_history["val_loss"][-1],
        "final_val_acc": combined_history["val_accuracy"][-1],
        "best_val_acc": max(stage1_metrics["best_val_acc"], stage2_metrics["best_val_acc"]),
        "test_loss": test_loss,
        "test_acc": test_acc,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "cm": cm,
        "report": report_dict,
        "report_text": report_text,
        "ext": ext,
        "cal_x": ext["cal_x"],
        "cal_y": ext["cal_y"],
        "stage1_checkpoint": str(stage1_ckpt),
        "stage2_checkpoint": str(stage2_ckpt),
    }
    return payload


results = globals().get("results", {})

resnet_stage1_epochs = 1 if FAST_RUN else 30
resnet_stage2_epochs = 1 if FAST_RUN else 18
results["resnet50"] = _run_two_stage_experiment(
    model_key="resnet50",
    backbone_name="resnet50",
    stage1_epochs=resnet_stage1_epochs,
    stage2_epochs=resnet_stage2_epochs,
    stage1_lr=2e-4,
    stage2_lr=2e-5,
    mixup_alpha=0.2,
    stage1_weight_decay=5e-5,
    stage2_weight_decay=5e-5,
    stage1_patience=12,
    stage2_patience=8,
    stage1_label_smoothing=0.08,
    stage2_label_smoothing=0.08,
    stage1_warmup=3,
    stage2_warmup=2,
 )

print("ResNet50 training complete.")
print(f"ResNet50 test accuracy: {results['resnet50']['test_acc']:.4f}")



Starting resnet50 experiment
Model params: total=24,691,012 | trainable=1,182,980


NameError: name 'fit_model' is not defined

In [7]:
# Section 5B: Train and Evaluate Quantum-Attention ResNet50

if "results" not in globals():
    results = {}

quantum_stage1_epochs = 1 if FAST_RUN else 42
quantum_stage2_epochs = 1 if FAST_RUN else 27
results["resnet50_quantum_attention"] = _run_two_stage_experiment(
    model_key="resnet50_quantum_attention",
    backbone_name="resnet50_quantum_attention",
    stage1_epochs=quantum_stage1_epochs,
    stage2_epochs=quantum_stage2_epochs,
    stage1_lr=1.6e-4,
    stage2_lr=1.5e-5,
    mixup_alpha=0.3,
    stage1_weight_decay=3e-5,
    stage2_weight_decay=2e-5,
    stage1_patience=14,
    stage2_patience=10,
    stage1_label_smoothing=0.10,
    stage2_label_smoothing=0.10,
    stage1_warmup=4,
    stage2_warmup=3,
)

print("Quantum-Attention training complete.")
print(f"Quantum-Attention test accuracy: {results['resnet50_quantum_attention']['test_acc']:.4f}")


Starting resnet50_quantum_attention experiment


Model params: total=25,743,684 | trainable=2,235,652
â–¶ resnet50_quantum_attention stage 1: 42 epochs | finetune=False | peak_lr=0.00016 | wd=3e-05 | ls=0.1 | mixup=0.3
[GPU][epoch 1 begin] 64, 1512, 4096


[GPU][epoch 1 end] 82, 1646, 4096 | loss=0.6708 | val_loss=0.4907 | acc=0.4149 | val_acc=0.6147


[GPU][epoch 2 begin] 82, 1646, 4096


[GPU][epoch 2 end] 90, 1646, 4096 | loss=0.5356 | val_loss=0.3684 | acc=0.5037 | val_acc=0.7012


...

[GPU][epoch 26 begin] 38, 2292, 4096


[GPU][epoch 26 end] 54, 2292, 4096 | loss=0.2046 | val_loss=0.1102 | acc=0.7505 | val_acc=0.9351


[GPU][epoch 27 begin] 54, 2292, 4096


[GPU][epoch 27 end] 75, 2292, 4096 | loss=0.2023 | val_loss=0.1137 | acc=0.7751 | val_acc=0.9360


Quantum-Attention training complete.
Quantum-Attention test accuracy: 0.8741


In [8]:
# Section 5C: Compare Both Models, Select Best, and Save Artifacts

def _generate_comprehensive_plots(results_dict: dict):
    for model_name, payload in results_dict.items():
        print(f"Generating plots for {model_name}...")
        plot_stem = "resnet50" if model_name == "resnet50" else "resnet50_quantum_attention"

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            payload["cm"],
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=MRI_CLASSES,
            yticklabels=MRI_CLASSES,
            ax=ax,
            cbar_kws={"label": "Count"},
        )
        ax.set_title(f"{model_name} - Confusion Matrix", fontweight="bold", fontsize=14)
        ax.set_xlabel("Predicted Class")
        ax.set_ylabel("True Class")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{plot_stem}_confusion_matrix.png", dpi=130, bbox_inches="tight")
        plt.close(fig)

        report = payload["report"]
        class_metrics = {cls: report[cls] for cls in MRI_CLASSES}
        fig, ax = plt.subplots(figsize=(12, 5))
        x = np.arange(len(MRI_CLASSES))
        width = 0.25
        precisions = [class_metrics[cls]["precision"] for cls in MRI_CLASSES]
        recalls = [class_metrics[cls]["recall"] for cls in MRI_CLASSES]
        f1s = [class_metrics[cls]["f1-score"] for cls in MRI_CLASSES]
        ax.bar(x - width, precisions, width, label="Precision", color="#3b82f6")
        ax.bar(x, recalls, width, label="Recall", color="#10b981")
        ax.bar(x + width, f1s, width, label="F1-Score", color="#f97316")
        ax.set_xlabel("Class")
        ax.set_ylabel("Score")
        ax.set_title(f"{model_name} - Per-Class Metrics", fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(MRI_CLASSES, rotation=15, ha="right")
        ax.legend()
        ax.grid(alpha=0.25, axis="y")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{plot_stem}_per_class_metrics.png", dpi=130, bbox_inches="tight")
        plt.close(fig)

        ext = payload["ext"]
        radar_keys = [k for k in ext.keys() if k not in {"cal_x", "cal_y"}]
        radar_vals = [float(ext[k]) for k in radar_keys]
        radar_angles = np.linspace(0, 2 * np.pi, len(radar_keys), endpoint=False).tolist()
        radar_vals += radar_vals[:1]
        radar_angles += radar_angles[:1]
        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(projection="polar"))
        ax.plot(radar_angles, radar_vals, "o-", linewidth=2, color="#8b5cf6")
        ax.fill(radar_angles, radar_vals, alpha=0.25, color="#8b5cf6")
        ax.set_xticks(radar_angles[:-1])
        ax.set_xticklabels(radar_keys, fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_title(f"{model_name} - Metrics Radar", fontweight="bold", pad=20)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{plot_stem}_metrics_radar.png", dpi=130, bbox_inches="tight")
        plt.close(fig)

        confidence = np.max(payload["y_prob"], axis=1)
        correct = (payload["y_pred"] == payload["y_true"]).astype(int)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(confidence[correct == 1], bins=20, alpha=0.65, label="Correct", color="#10b981", edgecolor="black")
        ax.hist(confidence[correct == 0], bins=20, alpha=0.65, label="Incorrect", color="#ef4444", edgecolor="black")
        ax.set_xlabel("Prediction Confidence")
        ax.set_ylabel("Count")
        ax.set_title(f"{model_name} - Confidence Distribution", fontweight="bold")
        ax.legend()
        ax.grid(alpha=0.25, axis="y")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{plot_stem}_confidence_dist.png", dpi=130, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(9, 8))
        ax.plot([0, 1], [0, 1], "k--", alpha=0.6, linewidth=2, label="Perfect calibration")
        ax.plot(payload["cal_x"], payload["cal_y"], "o-", linewidth=2.5, markersize=8, color="#8b5cf6", label="Model calibration")
        ax.fill_between(payload["cal_x"], payload["cal_y"], payload["cal_x"], alpha=0.2, color="#8b5cf6")
        ax.set_xlabel("Mean Predicted Probability")
        ax.set_ylabel("Fraction of Correct Predictions")
        ax.set_title(f"{model_name} - Calibration Curve", fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{plot_stem}_calibration_curve.png", dpi=130, bbox_inches="tight")
        plt.close(fig)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        y_true_bin = label_binarize(payload["y_true"], classes=np.arange(NUM_MRI_CLASSES))
        if y_true_bin.shape[1] != NUM_MRI_CLASSES:
            y_true_bin = np.pad(y_true_bin, ((0, 0), (0, NUM_MRI_CLASSES - y_true_bin.shape[1])))
        for i, cls_name in enumerate(MRI_CLASSES):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], payload["y_prob"][:, i])
            roc_auc = auc(fpr, tpr)
            axes[0].plot(fpr, tpr, label=f"{cls_name} (AUC={roc_auc:.3f})", linewidth=2)
        axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, linewidth=1.5)
        axes[0].set_xlabel("False Positive Rate")
        axes[0].set_ylabel("True Positive Rate")
        axes[0].set_title(f"{model_name} - ROC Curves")
        axes[0].legend(fontsize=8)
        axes[0].grid(alpha=0.25)

        class_accs = []
        for i in range(NUM_MRI_CLASSES):
            mask = payload["y_true"] == i
            class_accs.append(float(np.mean(payload["y_pred"][mask] == i)) if np.sum(mask) > 0 else 0.0)
        axes[1].bar(MRI_CLASSES, class_accs, color="#3b82f6", edgecolor="black", alpha=0.75)
        axes[1].set_ylabel("Per-Class Accuracy")
        axes[1].set_title(f"{model_name} - Per-Class Test Accuracy")
        axes[1].set_ylim(0, 1.1)
        axes[1].grid(alpha=0.25, axis="y")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"{plot_stem}_roc_and_class_accuracy.png", dpi=130, bbox_inches="tight")
        plt.close(fig)


def _save_both_models(results_dict: dict):
    for model_name, payload in results_dict.items():
        model = payload["model"]
        plot_stem = "resnet50" if model_name == "resnet50" else "resnet50_quantum_attention"
        save_path = SAVED_MODELS_DIR / f"{plot_stem}_best.pth"
        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(
            {
                "model_state_dict": state,
                "label_names": MRI_CLASSES,
                "model_kind": model_name,
                "test_accuracy": payload["test_acc"],
                "best_val_accuracy": payload["best_val_acc"],
            },
            save_path,
        )
        print(f"Saved {model_name} -> {save_path}")


def _print_model_rationale():
    rationale = """
ResNet50: strong baseline with stable transfer learning.
Quantum-Attention ResNet50: adds phase-modulated channel attention for richer MRI feature focus.
Both models use mixup, focal loss, warmup-cosine scheduling, and two-stage fine-tuning to improve generalization.
"""
    print(rationale)


if "resnet50" not in results or "resnet50_quantum_attention" not in results:
    raise RuntimeError("Run the ResNet and Quantum training cells before comparison.")

_generate_comprehensive_plots(results)
_save_both_models(results)

print("\n" + "=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)
for name, payload in results.items():
    print(f"\n{name}:")
    print(f"  Test Accuracy:      {payload['test_acc']:.4f} ({payload['test_acc'] * 100:.2f}%)")
    print(f"  Best Val Accuracy:  {payload['best_val_acc']:.4f} ({payload['best_val_acc'] * 100:.2f}%)")
    print(f"  F1-Score:           {payload['ext']['f1_score']:.4f}")
    print(f"  Sensitivity:        {payload['ext']['sensitivity']:.4f}")
    print(f"  Specificity:        {payload['ext']['specificity']:.4f}")
    print(f"  Calibration Error:  {payload['ext']['brier_calibration']:.4f}")

best_model_name = max(results.keys(), key=lambda k: (results[k]["best_val_acc"], results[k]["test_acc"]))
best_payload = results[best_model_name]

print("\n" + "=" * 80)
print(f"Selected best model: {best_model_name}")
print("=" * 80)
_print_model_rationale()

mri_model = best_payload["model"]
criterion = best_payload["criterion"]
mri_history = best_payload["history"]
history1 = best_payload["history1"]
history2 = best_payload["history2"]
stage1_metrics = best_payload["stage1_metrics"]
stage2_metrics = best_payload["stage2_metrics"]
mri_test_loss = best_payload["test_loss"]
mri_test_acc = best_payload["test_acc"]
y_true_mri = best_payload["y_true"]
y_pred_mri = best_payload["y_pred"]
y_prob_mri = best_payload["y_prob"]
cm = best_payload["cm"]
report = best_payload["report"]
report_text = best_payload["report_text"]
prec_mri = best_payload["ext"]["precision"]
rec_mri = best_payload["ext"]["recall"]
f1_mri = best_payload["ext"]["f1_score"]
final_train_loss = best_payload["final_train_loss"]
final_train_acc = best_payload["final_train_acc"]
final_val_loss = best_payload["final_val_loss"]
final_val_acc = best_payload["final_val_acc"]
test_loss = mri_test_loss
test_acc = mri_test_acc
best_val_acc = best_payload["best_val_acc"]
model = mri_model
model_kind = best_payload["model_kind"]
class_names = MRI_CLASSES
y_true = y_true_mri.tolist()
y_pred = y_pred_mri.tolist()
history = mri_history

best_state = {k: v.detach().cpu().clone() for k, v in mri_model.state_dict().items()}
torch.save(
    {
        "model_state_dict": best_state,
        "label_names": MRI_CLASSES,
        "model_kind": model_kind,
    },
    MRI_MODEL_PATH,
 )

mri_metrics = MetricsSummary(
    modality="MRI",
    final_train_loss=final_train_loss,
    final_train_acc=final_train_acc,
    final_val_loss=final_val_loss,
    final_val_acc=final_val_acc,
    test_loss=test_loss,
    test_acc=test_acc,
    precision=prec_mri,
    recall=rec_mri,
    f1_score=f1_mri,
    confusion_matrix=cm.tolist(),
    report=report,
 )

mri_summary = asdict(mri_metrics)
mri_report = {
    "MRI": mri_summary,
    "best_model": {
        "model_kind": model_kind,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
    },
    "dual_model_metrics": {},
    "Fusion": {
        "mri_mean_risk": float(np.mean(y_prob_mri @ MRI_RISK_WEIGHTS)),
    },
    "artifacts": {
        "plots": [
            str(PLOTS_DIR / "resnet50_training_curves.png"),
            str(PLOTS_DIR / "resnet50_quantum_attention_training_curves.png"),
            str(PLOTS_DIR / "dual_model_metrics_and_calibration.png"),
            str(PLOTS_DIR / "mri_risk_summary.png"),
        ],
        "models": [str(MRI_MODEL_PATH)],
        "metadata": [str(MRI_META_PATH)],
        "reports": [str(REPORT_JSON), str(SUMMARY_JSON), str(SUMMARY_CSV)],
    },
}

for name, payload in results.items():
    mri_report["dual_model_metrics"][name] = {
        **payload["ext"],
        "test_acc": payload["test_acc"],
        "best_val_acc": payload["best_val_acc"],
    }

selected_model_meta = {
    "class_names": MRI_CLASSES,
    "model_kind": model_kind,
    "selected_from": ["resnet50", "resnet50_quantum_attention"],
}
save_json(MRI_META_PATH, selected_model_meta)
save_json(SUMMARY_JSON, mri_report)
save_json(REPORT_JSON, mri_report)

summary_df = pd.DataFrame([
    {
        "modality": "MRI",
        **{k: v for k, v in mri_summary.items() if k != "report"},
        "report": json.dumps(mri_summary["report"], indent=2),
        "model_kind": model_kind,
    },
])
summary_df.to_csv(SUMMARY_CSV, index=False)

artifact_paths = mri_report["artifacts"]
print(f"\nFinal MRI accuracy: {mri_summary['test_acc']:.4f}")
print(f"Winner checkpoint saved to: {MRI_MODEL_PATH}")
print(f"All plots saved to: {PLOTS_DIR}")
print("Generated artifact paths:")
for category, paths in artifact_paths.items():
    for path in paths:
        print(f"- {path}")

Generating plots for resnet50...


Generating plots for resnet50_quantum_attention...


Saved resnet50 -> c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\saved_models\resnet50_best.pth
Saved resnet50_quantum_attention -> c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\saved_models\resnet50_quantum_attention_best.pth

MODEL PERFORMANCE COMPARISON

resnet50:
  Test Accuracy:      0.8170 (81.70%)
  Best Val Accuracy:  0.9126 (91.26%)
  F1-Score:           0.8187
  Sensitivity:        0.8170
  Specificity:        0.9234
  Calibration Error:  0.2046

resnet50_quantum_attention:
  Test Accuracy:      0.8741 (87.41%)
  Best Val Accuracy:  0.9360 (93.60%)
  F1-Score:           0.8742
  Sensitivity:        0.8741
  Specificity:        0.9460
  Calibration Error:  0.2183

Selected best model: resnet50_quantum_attention

ResNet50: strong baseline with stable transfer learning.
Quantum-Attention ResNet50: adds phase-modulated channel attention for richer MRI feature focus.
Both models use mixup, focal loss, warmup-cosine scheduling, and two-stage fine-tuning to improve generalizati


Final MRI accuracy: 0.8741
Winner checkpoint saved to: c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\saved_models\best_mri_model.pth
All plots saved to: c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\plots
Generated artifact paths:
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\plots\resnet50_training_curves.png
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\plots\resnet50_quantum_attention_training_curves.png
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\plots\dual_model_metrics_and_calibration.png
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\plots\mri_risk_summary.png
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\saved_models\best_mri_model.pth
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\saved_models\mri_metadata.json
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\reports\neuroscan_report.json
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\reports\neuroscan_metrics.json
- c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\reports\neuroscan_m

In [6]:
# Section 5D: Explicit per-model metric output (no retraining required)

required_models = ["resnet50", "resnet50_quantum_attention"]
metric_order = [
    ("Confidence", "confidence"),
    ("Sensitivity", "sensitivity"),
    ("Specificity", "specificity"),
    ("F1 score", "f1_score"),
    ("Macro average of F1 score", "f1_macro"),
    ("BRIER score", "brier_score"),
    ("Callibration of BRIER score", "brier_calibration"),
    ("Precisiion", "precision"),
    ("Recall", "recall"),
]


def _extract_metrics_for_model(model_name: str):
    # Priority 1: already-available in-memory results (if user trained in this session).
    if "results" in globals() and isinstance(results, dict) and model_name in results:
        ext = results[model_name].get("ext", {})
        return {k: ext.get(k, np.nan) for _, k in metric_order}

    # Priority 2: saved report from earlier completed training.
    if SUMMARY_JSON.exists():
        try:
            payload = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
            dual = payload.get("dual_model_metrics", {})
            if model_name in dual:
                src = dual[model_name]
                return {k: src.get(k, np.nan) for _, k in metric_order}
        except Exception:
            pass

    return {k: np.nan for _, k in metric_order}


rows = []
for model_name in required_models:
    extracted = _extract_metrics_for_model(model_name)
    row = {"Model": model_name}
    for label, key in metric_order:
        value = extracted.get(key, np.nan)
        row[label] = float(value) if value is not None and not pd.isna(value) else np.nan
    rows.append(row)

metrics_df = pd.DataFrame(rows)
print("\nExplicit per-model metric output (loaded from memory/report without retraining):")
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

if metrics_df.drop(columns=["Model"]).isna().all().all():
    print("\nNo saved metrics found yet. Run training once, then this cell can be reused without retraining.")

explicit_model_metrics = metrics_df.copy()



Explicit per-model metric output (loaded from memory/report without retraining):
                     Model  Confidence  Sensitivity  Specificity  F1 score  Macro average of F1 score  BRIER score  Callibration of BRIER score  Precisiion   Recall
                  resnet50    0.612305     0.817045     0.923418  0.818693                   0.798647     0.321455                     0.204555    0.825698 0.817045
resnet50_quantum_attention    0.655762     0.874120     0.945956  0.874185                   0.860562     0.258265                     0.218318    0.875017 0.874120


In [6]:
# Section 6: Ensure dual comparison artifacts are generated

if "results" not in globals() or not isinstance(results, dict):
    raise RuntimeError("Run Sections 5A, 5B, and 5C before Section 6.")

required_models = {"resnet50", "resnet50_quantum_attention"}
if not required_models.issubset(results.keys()):
    raise RuntimeError("Section 6 requires both model results from Sections 5A and 5B.")

# 1) Dual-model metrics + calibration plot (matches report artifact list)
metric_names = [
    "confidence",
    "sensitivity",
    "specificity",
    "f1_score",
    "f1_macro",
    "brier_score",
    "brier_calibration",
    "precision",
    "recall",
]
labels = list(results.keys())
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

x = np.arange(len(metric_names))
width = 0.35
vals0 = [results[labels[0]]["ext"][k] for k in metric_names]
vals1 = [results[labels[1]]["ext"][k] for k in metric_names]

axes[0].bar(x - width / 2, vals0, width=width, label=labels[0], color="#4f8ef7")
axes[0].bar(x + width / 2, vals1, width=width, label=labels[1], color="#10b981")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names, rotation=35, ha="right")
axes[0].set_title("Dual-Model Metric Comparison")
axes[0].legend()
axes[0].grid(alpha=0.25, axis="y")

axes[1].plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect calibration")
for name, payload in results.items():
    axes[1].plot(payload["cal_x"], payload["cal_y"], marker="o", linewidth=2, label=name)
axes[1].set_title("Calibration Curve (Confidence vs Accuracy)")
axes[1].set_xlabel("Mean predicted confidence")
axes[1].set_ylabel("Fraction of correct predictions")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "dual_model_metrics_and_calibration.png", dpi=130, bbox_inches="tight")
plt.close(fig)

# 2) MRI risk summary plot (matches report artifact list)
if "y_prob_mri" in globals():
    mean_risk = float(np.mean(y_prob_mri @ MRI_RISK_WEIGHTS))
else:
    best_model_name_local = max(results.keys(), key=lambda k: (results[k]["best_val_acc"], results[k]["test_acc"]))
    mean_risk = float(np.mean(results[best_model_name_local]["y_prob"] @ MRI_RISK_WEIGHTS))

fig, ax = plt.subplots(figsize=(8, 4))
labels_risk = ["MRI risk", "Complement"]
values = [mean_risk, 1.0 - mean_risk]
bars = ax.bar(labels_risk, values, color=["#4f8ef7", "#10b981"], edgecolor="none")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Risk Score")
ax.set_title("MRI Risk Summary", fontweight="bold")
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.03, f"{value:.3f}", ha="center", fontweight="bold")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "mri_risk_summary.png", dpi=130, bbox_inches="tight")
plt.close(fig)

print("Section 6 complete: dual comparison and MRI risk summary plots generated.")
print(f"- {PLOTS_DIR / 'dual_model_metrics_and_calibration.png'}")
print(f"- {PLOTS_DIR / 'mri_risk_summary.png'}")

RuntimeError: Run Sections 5A, 5B, and 5C before Section 6.

## Section 7: Interactive MRI Dashboard (PyTorch)

This dashboard shows the MRI-only inference pipeline, risk gauge, probability chart, and structured report.

- Upload MRI image (required)
- Images are converted to model-ready tensors (RGB, resized to 224x224, normalized with ImageNet stats)
- Outputs include risk gauge, MRI probability chart, explanation, and structured JSON report

In [7]:
os.environ["NS_LAUNCH_DASHBOARD"] = "1"

In [8]:
# =============================================================================
# Section 7A: MRI Dashboard Backend
#
# Fixes retained from the original dashboard implementation:
#  Fix 1  _to_rgb_uint8_array: float32 [0,1] numpy from Gradio webcam was
#          clipped to [0,255] then cast uint8 -> every pixel 0 or 1 -> black
#          image -> garbage output. Now detects float dtype and scales x255.
#  Fix 2  _prepare_input_tensor filepath branch: removed wasteful numpy
#          round-trip. read_image -> .float()/255 directly.
#  Fix 3  antialias kwarg wrapped in try/except for torchvision < 0.17.
#  Fix 4  _ensure_inference_model: isinstance(..., nn.Module) replaces
#          `is None` check which never triggered with a live session model.
#          Also forces .eval() on every call.
#  Fix 5  _predict_with_tta: .to(DEVICE) added after rotate() calls and the
#          model forward is wrapped in torch.autocast for AMP-safe inference.
# =============================================================================

import datetime

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

try:
    import gradio as gr
except ImportError as exc:
    raise ImportError("Gradio is required. pip install gradio>=4.0") from exc

try:
    from fpdf import FPDF
except ImportError as exc:
    raise ImportError("fpdf2 is required. pip install fpdf2") from exc


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 1. IMAGE PREPROCESSING
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _to_rgb_uint8_array(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.ndim == 3 and arr.shape[2] == 1:
        arr = np.repeat(arr, 3, axis=2)
    elif arr.ndim == 3 and arr.shape[2] > 3:
        arr = arr[:, :, :3]
    if arr.dtype in (np.float32, np.float64, float):
        arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    elif arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    return arr


def _prepare_input_tensor(image_input):
    if isinstance(image_input, (str, Path)):
        t = read_image(str(image_input))
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        elif t.shape[0] > 3:
            t = t[:3]
        x = t.float() / 255.0
    else:
        arr = _to_rgb_uint8_array(image_input)
        x = torch.from_numpy(arr).permute(2, 0, 1).contiguous().float() / 255.0

    try:
        x = TF.resize(x, [IMG_SIZE, IMG_SIZE], antialias=True)
    except TypeError:
        x = TF.resize(x, [IMG_SIZE, IMG_SIZE])

    x = TF.normalize(x, mean=list(IMAGENET_MEAN), std=list(IMAGENET_STD))
    return x.unsqueeze(0).to(DEVICE, non_blocking=True)


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 2. INFERENCE HELPERS
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _softmax_np(logits: torch.Tensor) -> np.ndarray:
    return torch.softmax(logits, dim=1).detach().cpu().numpy()[0]


def _predict_with_tta(model, image_input, class_names: list, passes: int = 8) -> dict:
    x = _prepare_input_tensor(image_input)
    model.eval()
    accum = None
    with torch.no_grad():
        for i in range(max(1, passes)):
            if i == 0:
                xi = x
            elif i % 3 == 1:
                xi = torch.flip(x, dims=[3])
            elif i % 3 == 2:
                xi = TF.rotate(x.squeeze(0), angle=5.0).unsqueeze(0).to(DEVICE)
            else:
                xi = TF.rotate(x.squeeze(0), angle=-5.0).unsqueeze(0).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(xi)
            p = _softmax_np(logits)
            accum = p if accum is None else accum + p

    avg = np.asarray(accum, dtype=np.float32) / float(max(1, passes))
    avg = np.nan_to_num(avg, nan=0.0, posinf=1.0, neginf=0.0)
    total = float(avg.sum())
    if total > 1e-8:
        avg /= total
    return {class_names[i]: float(avg[i]) for i in range(len(class_names))}


def _load_checkpoint_into_model(model, ckpt_path, fallback_classes):
    payload = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    if isinstance(payload, dict) and "model_state_dict" in payload:
        state = payload["model_state_dict"]
        classes = payload.get("label_names", fallback_classes)
    else:
        state = payload
        classes = fallback_classes
    model.load_state_dict(state, strict=False)
    model.to(DEVICE)
    model.eval()
    return classes


def _ensure_inference_model():
    global mri_model
    selected_backbone = "resnet50"
    if MRI_META_PATH.exists():
        try:
            meta = json.loads(MRI_META_PATH.read_text(encoding="utf-8"))
            selected_backbone = meta.get("backbone_name", meta.get("winner", selected_backbone))
        except Exception:
            selected_backbone = "resnet50"

    if not isinstance(globals().get("mri_model"), nn.Module):
        if selected_backbone == "hybrid_quantum_efficientnet":
            if not globals().get("PENNYLANE_AVAILABLE", False):
                raise RuntimeError("Winner model is hybrid_quantum_efficientnet but PennyLane is unavailable.")
            if "HybridQuantumEfficientNet" not in globals():
                raise RuntimeError("HybridQuantumEfficientNet is not defined in this kernel. Run Cells 11-14 first.")
            mri_model = HybridQuantumEfficientNet(NUM_MRI_CLASSES, N_QUBITS, N_Q_LAYERS).to(DEVICE)
        elif selected_backbone in {"resnet50", "resnet50_quantum_attention"}:
            mri_model = build_model(selected_backbone, NUM_MRI_CLASSES)
        else:
            mri_model = build_model("resnet50", NUM_MRI_CLASSES)
        _load_checkpoint_into_model(mri_model, MRI_MODEL_PATH, MRI_CLASSES)

    mri_model.eval()


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 3. PREDICTION FUNCTIONS
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _risk_band(score: float):
    if score < 0.20:
        return "Very Low", "#16a34a", "NO SIGNIFICANT SIGNAL"
    if score < 0.40:
        return "Low", "#65a30d", "LOW SIGNAL"
    if score < 0.60:
        return "Moderate", "#d97706", "MODERATE SIGNAL"
    if score < 0.80:
        return "High", "#dc2626", "HIGH SIGNAL"
    return "Very High", "#7f1d1d", "STRONG SIGNAL - REFER"


def predict_mri_dashboard(image_input, use_tta: bool = True) -> dict:
    _ensure_inference_model()
    passes = MRI_TTA_PASSES if use_tta else 1
    probs = _predict_with_tta(mri_model, image_input, MRI_CLASSES, passes=passes)
    label = max(probs, key=probs.get)

    healthy_idx = int(np.argmin(MRI_RISK_WEIGHTS))
    healthy_class = MRI_CLASSES[healthy_idx]
    healthy_prob = float(probs.get(healthy_class, 0.0))
    risk = float(np.clip(1.0 - healthy_prob, 0.0, 1.0))

    return {
        "label": label,
        "confidence": float(probs[label]),
        "probs": probs,
        "healthy_class": healthy_class,
        "healthy_prob": healthy_prob,
        "risk": risk,
    }


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 4. PLOTLY FIGURES
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _gauge_fig(score: float, band: str, color: str) -> go.Figure:
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=round(score * 100, 1),
        title={"text": f"<b>{band} Risk</b>", "font": {"size": 15}},
        number={"suffix": "%", "font": {"size": 24}},
        gauge={
            "axis": {"range": [0, 100], "tickfont": {"size": 10}},
            "bar": {"color": color, "thickness": 0.22},
            "steps": [
                {"range": [0, 20], "color": "#dcfce7"},
                {"range": [20, 40], "color": "#ecfccb"},
                {"range": [40, 60], "color": "#fef3c7"},
                {"range": [60, 80], "color": "#fee2e2"},
                {"range": [80, 100], "color": "#fecaca"},
            ],
            "threshold": {"line": {"color": color, "width": 3}, "thickness": 0.75, "value": score * 100},
        },
    ))
    fig.update_layout(height=300, margin=dict(l=20, r=20, t=55, b=15),
                      paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                      font={"color": "#e2e8f0"})
    return fig


def _mri_radar_fig(probs: dict, classes: list) -> go.Figure:
    vals = [probs.get(c, 0.0) for c in classes]
    fig = go.Figure(go.Scatterpolar(
        r=vals + [vals[0]], theta=classes + [classes[0]], fill="toself",
        line=dict(color="#38bdf8", width=2), fillcolor="rgba(56,189,248,0.18)",
    ))
    fig.update_layout(
        polar=dict(bgcolor="rgba(0,0,0,0)",
                   radialaxis=dict(range=[0, 1], gridcolor="#334155", tickfont={"size": 9}),
                   angularaxis=dict(gridcolor="#334155")),
        height=320, margin=dict(l=30, r=30, t=30, b=30),
        paper_bgcolor="rgba(0,0,0,0)", font={"color": "#e2e8f0"}, showlegend=False,
    )
    return fig


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 5. HTML PANEL BUILDERS
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_PANEL = (
    "<div style='border:1px solid #334155;border-radius:14px;padding:12px 14px;"
    "background:rgba(15,23,42,0.65);'>{}</div>"
 )


def _badge_html(mri_sig: dict) -> str:
    band, color, tag = _risk_band(mri_sig["risk"])
    return (
        f"<div style='padding:14px 18px;border-radius:12px;border:2px solid {color};"
        f"background:{color}22;'>"
        f"<div style='font-size:1.25rem;font-weight:800;color:{color};'>{tag}</div>"
        f"<div style='font-size:0.85rem;color:#94a3b8;margin-top:4px;'>"
        f"{band} Risk Â· Score {mri_sig['risk']*100:.1f}%</div>"
        f"<div style='font-size:0.75rem;color:#64748b;margin-top:6px;'>"
        f"Screening aid only - not a clinical diagnosis</div>"
        f"</div>"
    )


def _summary_html(mri_sig: dict) -> str:
    top_classes = sorted(mri_sig["probs"].items(), key=lambda x: -x[1])[:3]
    rows = ""
    for idx, (name, score) in enumerate(top_classes):
        w = int(120 * score)
        rows += (
            f"<div style='margin-bottom:5px'>"
            f"<div style='display:flex;justify-content:space-between;font-size:0.8rem'>"
            f"<span>{idx+1}. {name}</span><strong>{score:.1%}</strong></div>"
            f"<div style='height:6px;background:#1e293b;border-radius:4px'>"
            f"<div style='height:6px;width:{w}px;background:#38bdf8;border-radius:4px'></div></div>"
            f"</div>"
        )
    return _PANEL.format(
        f"<div style='display:flex;gap:14px;flex-wrap:wrap'>"
        f"<div style='flex:1;min-width:180px'>"
        f"<div style='font-size:0.78rem;color:#64748b;margin-bottom:6px'>MRI TOP CLASSES</div>"
        f"{rows}</div>"
        f"<div style='flex:1;min-width:180px'>"
        f"<div style='font-size:0.78rem;color:#64748b;margin-bottom:6px'>RISK SUMMARY</div>"
        f"<div style='font-size:1rem;color:#e2e8f0'>"
        f"Predicted: <b>{mri_sig['label']}</b><br>"
        f"Confidence: <b>{mri_sig['confidence']:.1%}</b><br>"
        f"Healthy baseline: <b>{mri_sig['healthy_class']}</b> ({mri_sig['healthy_prob']:.1%})<br>"
        f"MRI risk score: <b>{mri_sig['risk']:.4f}</b></div></div>"
        f"</div>"
    )


def _mri_html(mri_sig: dict) -> str:
    rows = [f"<b>Predicted Stage:</b> {mri_sig['label']} ({mri_sig['confidence']:.1%})<br>"]
    rows += [f"<b>Healthy Baseline:</b> {mri_sig['healthy_class']} ({mri_sig['healthy_prob']:.1%})<br>"]
    rows += [f"<b>MRI Risk Score:</b> {mri_sig['risk']:.4f}<br><br>"]
    for cls, p in sorted(mri_sig["probs"].items(), key=lambda x: -x[1]):
        w = int(160 * p)
        rows.append(
            f"<div style='font-size:0.82rem;margin-bottom:3px'>{cls}: {p:.1%}</div>"
            f"<div style='height:7px;background:#1e293b;border-radius:4px;margin-bottom:5px'>"
            f"<div style='height:7px;width:{w}px;background:#38bdf8;border-radius:4px'></div></div>"
        )
    return _PANEL.format("\n".join(rows))


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 6. EXPLANATION TEXT
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def explain_dashboard(mri_sig: dict) -> str:
    lines = [
        "NeuroScan AI - Clinical Screening Explanation",
        "-" * 54,
        "DISCLAIMER: This output is a research screening aid and",
        "does NOT constitute a clinical diagnosis.",
        "",
        f"Overall Risk Band : {_risk_band(mri_sig['risk'])[0]}",
        f"Risk Score        : {mri_sig['risk']:.4f} ({mri_sig['risk']*100:.1f}%)",
        "",
        "--- MRI Analysis ---",
        f"Predicted stage   : {mri_sig['label']} (confidence {mri_sig['confidence']:.1%})",
        f"Healthy baseline  : {mri_sig['healthy_class']} ({mri_sig['healthy_prob']:.1%})",
        f"MRI risk score    : 1 - P(healthy) = {mri_sig['risk']:.4f}",
        "",
        "MRI Class Probabilities:",
    ]
    for cls, p in sorted(mri_sig['probs'].items(), key=lambda x: -x[1]):
        lines.append(f"  {cls:<30s} {p:.4f}  ({p:.1%})")
    lines += [
        "",
        "--- MRI-only note ---",
        "This dashboard evaluates MRI evidence only and writes a MRI-only report.",
    ]
    return "\n".join(lines)


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 7. PDF REPORT
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _pdf_safe(text: str) -> str:
    return str(text).encode("ascii", "replace").decode("ascii")


class MRIReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.set_text_color(10, 35, 80)
        self.cell(0, 9, "NeuroScan MRI - Alzheimer Screening Report", ln=True, align="C")
        self.set_font("Helvetica", "", 7.5)
        self.set_text_color(160, 40, 40)
        self.cell(0, 5,
                  "RESEARCH SCREENING AID ONLY - NOT A CLINICAL DIAGNOSIS. "
                  "Consult a qualified neurologist for medical advice.",
                  ln=True, align="C")
        self.set_text_color(90, 90, 90)
        self.ln(1)

    def footer(self):
        self.set_y(-12)
        self.set_font("Helvetica", "", 7)
        self.set_text_color(150, 150, 150)
        self.cell(0, 5, f"NeuroScan MRI | Page {self.page_no()} | Research Use Only", align="C")

    def section(self, title: str):
        self.set_fill_color(224, 236, 255)
        self.set_text_color(20, 20, 90)
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 7, f"  {_pdf_safe(title)}", ln=True, fill=True)
        self.set_text_color(30, 30, 30)
        self.ln(1)


def generate_pdf_report(mri_sig, explanation, patient_name="", patient_age="", notes="") -> str:
    reports_dir = Path(PROJECT_ROOT) / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)

    pdf = MRIReport()
    pdf.add_page()

    pdf.section("1. Patient Information")
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 6, _pdf_safe(f"Name: {patient_name or 'Anonymous'}"), ln=True)
    pdf.cell(0, 6, _pdf_safe(f"Age:  {patient_age or 'Not provided'}"), ln=True)
    pdf.cell(0, 6, _pdf_safe(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"), ln=True)
    pdf.ln(2)

    pdf.section("2. MRI Analysis")
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 6, _pdf_safe(f"Predicted Stage   : {mri_sig['label']} ({mri_sig['confidence']:.1%})"), ln=True)
    pdf.cell(0, 6, _pdf_safe(f"Healthy Baseline  : {mri_sig['healthy_class']} ({mri_sig['healthy_prob']:.1%})"), ln=True)
    pdf.cell(0, 6, _pdf_safe(f"MRI Risk Score    : {mri_sig['risk']:.4f}"), ln=True)
    pdf.ln(1)
    pdf.set_font("Helvetica", "B", 9)
    pdf.cell(0, 5, "Class Probabilities:", ln=True)
    pdf.set_font("Helvetica", "", 9)
    for cls, p in sorted(mri_sig['probs'].items(), key=lambda x: -x[1]):
        pdf.cell(0, 5, _pdf_safe(f"  {cls:<32} {p:.4f}  ({p:.1%})"), ln=True)
    pdf.ln(2)

    pdf.section("3. Decision Explanation")
    pdf.set_font("Courier", "", 7.5)
    for line in explanation.split("\n"):
        txt = _pdf_safe(line[:140])
        pdf.cell(0, 4.2, txt if txt.strip() else " ", ln=True)

    if notes and notes.strip():
        pdf.ln(1)
        pdf.section("4. Clinical Notes")
        pdf.set_font("Helvetica", "", 10)
        for line in str(notes).split("\n"):
            pdf.cell(0, 5.5, _pdf_safe(line[:140]) or " ", ln=True)

    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    out = reports_dir / f"neuroscan_dashboard_report_{stamp}.pdf"
    pdf.output(str(out))
    return str(out)


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 8. MAIN CALLBACK
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _error_payload(exc):
    msg = str(exc)
    err_html = f"<div style='color:#ef4444;font-weight:700;padding:12px'>Dashboard error: {msg}</div>"
    empty = go.Figure()
    empty.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return err_html, err_html, err_html, err_html, {}, empty, empty, None


def run_analysis_dashboard(mri_img, use_tta, patient_name, patient_age, notes):
    if mri_img is None:
        err = ("<span style='color:#ef4444;font-weight:700'>"
               "Please upload an MRI scan to begin analysis.</span>")
        empty = go.Figure()
        return err, err, err, "", {}, empty, empty, None

    try:
        mri_sig = predict_mri_dashboard(mri_img, use_tta=bool(use_tta))
        explanation = explain_dashboard(mri_sig)

        report_json = {
            "mri": {
                k: mri_sig[k]
                for k in ("label", "confidence", "risk", "healthy_class", "healthy_prob")
            },
            "mri_probs": mri_sig["probs"],
            "risk_band": _risk_band(mri_sig["risk"])[0],
            "risk_score": mri_sig["risk"],
        }

        pdf_path = generate_pdf_report(
            mri_sig, explanation,
            patient_name=patient_name, patient_age=patient_age, notes=notes,
        )

        return (
            _badge_html(mri_sig),
            _summary_html(mri_sig),
            _mri_html(mri_sig),
            explanation,
            report_json,
            _mri_radar_fig(mri_sig["probs"], MRI_CLASSES),
            _gauge_fig(mri_sig["risk"], _risk_band(mri_sig["risk"])[0], _risk_band(mri_sig["risk"])[1]),
            pdf_path,
        )

    except Exception as exc:
        return _error_payload(exc)


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 9. CSS
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

NEUROSCAN_CSS = """
.gradio-container {
  background:
    radial-gradient(circle at 8% 8%,  #172554 0%, transparent 35%),
    radial-gradient(circle at 92% 10%, #0f766e 0%, transparent 40%),
    linear-gradient(140deg, #050b18 0%, #0b1220 55%, #111827 100%) !important;
  color: #e2e8f0 !important;
  font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important;
}
#ns-shell { max-width: 1500px; margin: 0 auto; padding: 16px 20px 40px; }
.ns-hdr {
  border: 1px solid #334155;
  background: linear-gradient(120deg, rgba(30,41,59,.78), rgba(15,23,42,.78));
  border-radius: 18px; padding: 20px 26px; margin-bottom: 16px;
}
.ns-hdr-logo   { display:flex; align-items:center; gap:12px; margin-bottom:6px; }
.ns-hdr-badge  { font-size:.70rem; font-weight:700; letter-spacing:.12em;
                 color:#0f172a; background:#38bdf8; border-radius:6px;
                 padding:3px 8px; text-transform:uppercase; }
.ns-hdr-title  { font-size:2rem; font-weight:800; color:#67e8f9; letter-spacing:.3px; }
.ns-hdr-sub    { color:#93c5fd; margin-top:4px; font-size:.92rem; }
.ns-disclaimer {
  margin-top:10px; padding:8px 12px; border-radius:8px;
  background:rgba(239,68,68,0.10); border:1px solid rgba(239,68,68,0.35);
  color:#fca5a5; font-size:.78rem;
}
.card { border:1px solid #334155; border-radius:14px; padding:12px 14px;
        background:rgba(15,23,42,0.65); color:#e2e8f0; }
button.primary { background:#0ea5e9 !important; color:#fff !important; font-weight:700 !important; }
button.primary:hover { background:#0284c7 !important; }
"""

print("Dashboard backend loaded.")
print("Run the next cell to launch the Gradio interface.")


Dashboard backend loaded.
Run the next cell to launch the Gradio interface.


In [9]:
# Section 7A+ : Dashboard confidence and Grad-CAM enhancements

# Keep track of which checkpoint is loaded so dashboard inference is never random after restart.
_DASHBOARD_MODEL_CACHE = globals().get("_DASHBOARD_MODEL_CACHE", {"token": None})
_ACTIVE_DASHBOARD_MODEL_INFO = globals().get("_ACTIVE_DASHBOARD_MODEL_INFO", {})


def _safe_read_json(path: Path) -> dict:
    try:
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        pass
    return {}


def _safe_checkpoint_model_kind(ckpt_path: Path):
    try:
        payload = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if isinstance(payload, dict):
            return payload.get("model_kind")
    except Exception:
        pass
    return None


def _resolve_best_model_context() -> dict:
    allowed = {"resnet50", "resnet50_quantum_attention"}

    meta = _safe_read_json(MRI_META_PATH)
    for key in ("model_kind", "backbone_name", "winner"):
        candidate = meta.get(key)
        if candidate in allowed:
            return {"model_kind": candidate, "selection_source": f"metadata:{key}", "reference_test_acc": None}

    for report_path in (REPORT_JSON, SUMMARY_JSON):
        report_payload = _safe_read_json(report_path)
        best_block = report_payload.get("best_model", {}) if isinstance(report_payload, dict) else {}
        candidate = best_block.get("model_kind")
        if candidate in allowed:
            return {
                "model_kind": candidate,
                "selection_source": f"report:{report_path.name}",
                "reference_test_acc": best_block.get("test_acc"),
            }

    candidate = _safe_checkpoint_model_kind(MRI_MODEL_PATH)
    if candidate in allowed:
        return {"model_kind": candidate, "selection_source": "checkpoint:model_kind", "reference_test_acc": None}

    return {"model_kind": "resnet50", "selection_source": "fallback:resnet50", "reference_test_acc": None}


def _ensure_inference_model():
    global mri_model, MRI_CLASSES, _ACTIVE_DASHBOARD_MODEL_INFO

    if not MRI_MODEL_PATH.exists():
        raise RuntimeError(f"Checkpoint not found: {MRI_MODEL_PATH}. Train once (5A/5B/5C) or copy the saved model.")

    model_ctx = _resolve_best_model_context()
    selected_backbone = model_ctx["model_kind"]

    model_ok = isinstance(globals().get("mri_model"), nn.Module)
    model_kind = getattr(globals().get("mri_model", None), "backbone_name", None)
    token = f"{selected_backbone}|{str(MRI_MODEL_PATH)}|{model_ctx.get('selection_source', '')}"
    needs_reload = (not model_ok) or (model_kind != selected_backbone) or (_DASHBOARD_MODEL_CACHE.get("token") != token)

    if needs_reload:
        mri_model = build_model(selected_backbone, NUM_MRI_CLASSES)
        loaded_classes = _load_checkpoint_into_model(mri_model, MRI_MODEL_PATH, MRI_CLASSES)
        if isinstance(loaded_classes, (list, tuple)) and len(loaded_classes) > 0:
            MRI_CLASSES = list(loaded_classes)
        _DASHBOARD_MODEL_CACHE["token"] = token

    mri_model.eval()
    _ACTIVE_DASHBOARD_MODEL_INFO = {
        "model_kind": selected_backbone,
        "selection_source": model_ctx.get("selection_source", "unknown"),
        "reference_test_acc": model_ctx.get("reference_test_acc"),
    }


def _top2_gap(probs: dict):
    ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)
    if not ranked:
        return None, None, 0.0, False
    top1_name, top1_prob = ranked[0]
    top2_prob = ranked[1][1] if len(ranked) > 1 else 0.0
    gap = float(top1_prob - top2_prob)
    low_conf = float(top1_prob) < 0.60
    return top1_name, float(top1_prob), gap, low_conf


def _hero_signal_html(mri_sig: dict) -> str:
    band, color, tag = _risk_band(mri_sig["risk"])
    top1_name, top1_prob, gap, low_conf = _top2_gap(mri_sig["probs"])
    warn_html = ""
    if low_conf:
        warn_html = (
            "<div style='margin-top:10px;padding:10px 12px;border-radius:10px;"
            "background:rgba(245,158,11,.18);border:1px solid rgba(245,158,11,.55);"
            "color:#fbbf24;font-weight:700'>"
            "Low-confidence prediction: manually review MRI and clinical context."
            "</div>"
        )
    return (
        f"<div style='padding:20px 22px;border-radius:16px;border:2px solid {color};"
        f"background:linear-gradient(110deg,{color}22,rgba(15,23,42,.78));min-height:180px'>"
        f"<div style='font-size:2rem;font-weight:900;letter-spacing:.02em;color:{color};line-height:1'>{tag}</div>"
        f"<div style='margin-top:8px;font-size:1.02rem;color:#cbd5e1'>"
        f"{band} Risk | Score {mri_sig['risk']*100:.1f}%"
        f"</div>"
        "<div style='display:grid;grid-template-columns:repeat(3,minmax(120px,1fr));gap:10px;margin-top:14px'>"
        f"<div style='background:rgba(2,6,23,.45);border:1px solid #334155;border-radius:10px;padding:10px'>"
        f"<div style='font-size:.74rem;color:#94a3b8'>TOP CLASS</div><div style='font-weight:800;color:#e2e8f0'>{top1_name}</div></div>"
        f"<div style='background:rgba(2,6,23,.45);border:1px solid #334155;border-radius:10px;padding:10px'>"
        f"<div style='font-size:.74rem;color:#94a3b8'>TOP CONFIDENCE</div><div style='font-weight:800;color:#e2e8f0'>{top1_prob:.1%}</div></div>"
        f"<div style='background:rgba(2,6,23,.45);border:1px solid #334155;border-radius:10px;padding:10px'>"
        f"<div style='font-size:.74rem;color:#94a3b8'>TOP-2 GAP (p1-p2)</div><div style='font-weight:800;color:#e2e8f0'>{gap:.4f}</div></div>"
        "</div>"
        f"{warn_html}"
        "<div style='font-size:.82rem;color:#94a3b8;margin-top:10px'>"
        "Screening aid only - not a clinical diagnosis"
        "</div>"
        "</div>"
    )


def _model_trace_html() -> str:
    info = globals().get("_ACTIVE_DASHBOARD_MODEL_INFO", {}) or {}
    model_kind = info.get("model_kind", getattr(globals().get("mri_model", None), "backbone_name", "unknown"))
    source = info.get("selection_source", "runtime")
    test_acc = info.get("reference_test_acc")
    acc_line = f"<div style='font-size:.84rem;color:#cbd5e1'>Reference test accuracy: <b>{float(test_acc):.4f}</b></div>" if test_acc is not None else ""
    return (
        "<div style='border:1px solid #334155;border-radius:12px;padding:10px 12px;"
        "background:rgba(2,6,23,.45);margin-top:10px'>"
        "<div style='font-size:.78rem;color:#94a3b8;margin-bottom:4px'>MODEL USED BY DASHBOARD</div>"
        f"<div style='font-size:.90rem;color:#e2e8f0'>Backbone: <b>{model_kind}</b></div>"
        f"<div style='font-size:.78rem;color:#94a3b8'>Selection source: {source}</div>"
        f"{acc_line}"
        "</div>"
    )


def _confidence_panel_html(mri_sig: dict) -> str:
    top1_name, top1_prob, gap, low_conf = _top2_gap(mri_sig["probs"])
    warn_html = ""
    if low_conf:
        warn_html = (
            "<div style='margin-top:8px;padding:8px 10px;border-radius:8px;"
            "background:rgba(245,158,11,.15);border:1px solid rgba(245,158,11,.45);"
            "color:#fbbf24;font-weight:700'>"
            "Low-confidence prediction, review manually."
            "</div>"
        )
    return (
        "<div style='border:1px solid #334155;border-radius:12px;padding:10px 12px;"
        "background:rgba(15,23,42,0.65);margin-top:10px'>"
        "<div style='font-size:.78rem;color:#94a3b8;margin-bottom:4px'>CONFIDENCE CHECK</div>"
        f"<div style='font-size:.88rem;color:#e2e8f0'>Top class: <b>{top1_name}</b> ({top1_prob:.1%})</div>"
        f"<div style='font-size:.88rem;color:#e2e8f0'>Top-2 gap (p1 - p2): <b>{gap:.4f}</b></div>"
        f"{warn_html}"
        "</div>"
    )


def _per_class_explanation_html(mri_sig: dict) -> str:
    ranked = sorted(mri_sig["probs"].items(), key=lambda kv: kv[1], reverse=True)
    healthy_prob = float(mri_sig["healthy_prob"])
    rows = []
    for rank, (name, prob) in enumerate(ranked, start=1):
        delta = float(prob - healthy_prob)
        delta_color = "#22c55e" if delta >= 0 else "#ef4444"
        rows.append(
            "<tr>"
            f"<td style='padding:4px 6px;color:#93c5fd'>#{rank}</td>"
            f"<td style='padding:4px 6px;color:#e2e8f0'>{name}</td>"
            f"<td style='padding:4px 6px;color:#e2e8f0'>{prob:.1%}</td>"
            f"<td style='padding:4px 6px;color:{delta_color}'>{delta:+.4f}</td>"
            "</tr>"
        )
    return (
        "<div style='border:1px solid #334155;border-radius:12px;padding:10px 12px;"
        "background:rgba(15,23,42,0.65)'>"
        "<div style='font-size:.78rem;color:#94a3b8;margin-bottom:6px'>PER-CLASS EXPLANATION</div>"
        "<div style='font-size:.84rem;color:#cbd5e1;margin-bottom:8px'>"
        "Delta vs healthy baseline = class probability - P(healthy)."
        "</div>"
        "<table style='width:100%;border-collapse:collapse;font-size:.82rem'>"
        "<thead><tr style='color:#94a3b8;text-align:left'>"
        "<th style='padding:4px 6px'>Rank</th>"
        "<th style='padding:4px 6px'>Class</th>"
        "<th style='padding:4px 6px'>Probability</th>"
        "<th style='padding:4px 6px'>Delta vs healthy</th>"
        "</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table></div>"
    )


def _find_last_conv(module: nn.Module):
    last_conv = None
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    return last_conv


def _pick_gradcam_target_module(model: nn.Module):
    if hasattr(model, "backbone") and isinstance(model.backbone, nn.Module):
        target = _find_last_conv(model.backbone)
        if target is not None:
            return target
    if hasattr(model, "stem") and isinstance(model.stem, nn.Module):
        target = _find_last_conv(model.stem)
        if target is not None:
            return target
    target = _find_last_conv(model)
    if target is None:
        raise RuntimeError("Could not locate a convolution layer for Grad-CAM.")
    return target


def _preview_rgb_uint8(image_input):
    if isinstance(image_input, (str, Path)):
        t = read_image(str(image_input))
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        elif t.shape[0] > 3:
            t = t[:3]
        arr = t.permute(1, 2, 0).contiguous().cpu().numpy().astype(np.uint8)
    else:
        arr = _to_rgb_uint8_array(image_input)
    return arr


def _normalize_map(m: np.ndarray):
    m = np.asarray(m, dtype=np.float32)
    m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = float(np.min(m)), float(np.max(m))
    if hi - lo < 1e-8:
        return np.zeros_like(m, dtype=np.float32)
    return (m - lo) / (hi - lo)


def _compose_gradcam_triptych(preview: np.ndarray, heat: np.ndarray, overlay: np.ndarray) -> np.ndarray:
    h, w = preview.shape[:2]
    gap = max(6, w // 60)
    canvas = np.full((h, w * 3 + gap * 4, 3), 8, dtype=np.uint8)
    x0 = gap
    x1 = x0 + w + gap
    x2 = x1 + w + gap
    canvas[:, x0:x0 + w, :] = preview
    canvas[:, x1:x1 + w, :] = heat
    canvas[:, x2:x2 + w, :] = overlay
    return canvas


def _compute_gradcam_overlay(model: nn.Module, image_input, class_names: list):
    _ = class_names
    x = _prepare_input_tensor(image_input).detach()
    preview = _preview_rgb_uint8(image_input)
    target_module = _pick_gradcam_target_module(model)

    activations = {}
    gradients = {}

    def _fwd_hook(_mod, _inp, out):
        activations["value"] = out

    def _bwd_hook(_mod, _gin, gout):
        if gout and gout[0] is not None:
            gradients["value"] = gout[0]

    h1 = target_module.register_forward_hook(_fwd_hook)
    h2 = target_module.register_full_backward_hook(_bwd_hook)

    cam_map = None
    saliency_map = None
    try:
        x.requires_grad_(True)
        model.zero_grad(set_to_none=True)
        logits = model(x)
        pred_idx = int(logits.argmax(dim=1).item())
        score = logits[:, pred_idx].sum()
        score.backward(retain_graph=False)

        if "value" in activations and "value" in gradients:
            acts = activations["value"].detach()
            grads = gradients["value"].detach()
            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam = torch.relu((weights * acts).sum(dim=1, keepdim=True))
            cam = F.interpolate(cam, size=preview.shape[:2], mode="bilinear", align_corners=False)
            cam_map = cam[0, 0].detach().cpu().numpy()

        if x.grad is not None:
            saliency_map = x.grad.detach().abs().mean(dim=1)[0].cpu().numpy()
    finally:
        h1.remove()
        h2.remove()

    if cam_map is None:
        cam_map = saliency_map if saliency_map is not None else np.zeros(preview.shape[:2], dtype=np.float32)

    cam_map = _normalize_map(cam_map)
    heat = (plt.get_cmap("turbo")(cam_map)[..., :3] * 255).astype(np.uint8)
    overlay = (0.55 * preview.astype(np.float32) + 0.45 * heat.astype(np.float32)).clip(0, 255).astype(np.uint8)
    return _compose_gradcam_triptych(preview, heat, overlay)


# Override with enhanced output contract (adds per-class panel + Grad-CAM image)
def _error_payload(exc):
    msg = str(exc)
    err_html = f"<div style='color:#ef4444;font-weight:700;padding:12px'>Dashboard error: {msg}</div>"
    empty = go.Figure()
    empty.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    return err_html, err_html, err_html, err_html, err_html, {}, empty, empty, None, None


def run_analysis_dashboard(mri_img, use_tta, patient_name, patient_age, notes):
    if mri_img is None:
        err = (
            "<span style='color:#ef4444;font-weight:700'>"
            "Please upload an MRI scan to begin analysis.</span>"
        )
        empty = go.Figure()
        return err, err, err, err, "", {}, empty, empty, None, None

    try:
        mri_sig = predict_mri_dashboard(mri_img, use_tta=bool(use_tta))
        explanation = explain_dashboard(mri_sig)

        active_info = globals().get("_ACTIVE_DASHBOARD_MODEL_INFO", {}) or {}
        report_json = {
            "mri": {
                k: mri_sig[k]
                for k in ("label", "confidence", "risk", "healthy_class", "healthy_prob")
            },
            "mri_probs": mri_sig["probs"],
            "risk_band": _risk_band(mri_sig["risk"])[0],
            "risk_score": mri_sig["risk"],
            "model_used": {
                "model_kind": active_info.get("model_kind", getattr(mri_model, "backbone_name", "unknown")),
                "selection_source": active_info.get("selection_source", "runtime"),
                "reference_test_acc": active_info.get("reference_test_acc"),
            },
        }

        top1_name, top1_prob, gap, low_conf = _top2_gap(mri_sig["probs"])
        report_json["confidence_check"] = {
            "top_class": top1_name,
            "top_class_confidence": top1_prob,
            "top2_gap": gap,
            "low_confidence_warning": low_conf,
            "threshold": 0.60,
        }

        hero_html = _hero_signal_html(mri_sig)
        summary_html = _summary_html(mri_sig) + _model_trace_html() + _confidence_panel_html(mri_sig)
        class_explain_html = _per_class_explanation_html(mri_sig)

        gradcam_overlay = None
        gradcam_status = "ok"
        try:
            gradcam_overlay = _compute_gradcam_overlay(mri_model, mri_img, MRI_CLASSES)
        except Exception:
            # Keep Grad-CAM as optional visualization; prediction outputs must still succeed.
            gradcam_status = "unavailable"
            gradcam_overlay = None
        report_json["gradcam_status"] = gradcam_status

        pdf_path = generate_pdf_report(
            mri_sig,
            explanation,
            patient_name=patient_name,
            patient_age=patient_age,
            notes=notes,
        )

        return (
            hero_html,
            summary_html,
            _mri_html(mri_sig),
            class_explain_html,
            explanation,
            report_json,
            _mri_radar_fig(mri_sig["probs"], MRI_CLASSES),
            _gauge_fig(mri_sig["risk"], _risk_band(mri_sig["risk"])[0], _risk_band(mri_sig["risk"])[1]),
            gradcam_overlay,
            pdf_path,
        )

    except Exception as exc:
        return _error_payload(exc)


print("Section 7A+ enhancements loaded: dashboard now auto-selects the best available model and reports which model was used.")

Section 7A+ enhancements loaded: dashboard now auto-selects the best available model and reports which model was used.


In [10]:
# Section 7A++: Model Verification (before dashboard launch)

if "_resolve_best_model_context" not in globals() or "_ensure_inference_model" not in globals():
    raise RuntimeError("Run Cell 15 (Section 7A+) before this verification cell.")

ctx = _resolve_best_model_context()
_ensure_inference_model()

model_kind = ctx.get("model_kind", getattr(mri_model, "backbone_name", "unknown"))
source = ctx.get("selection_source", "unknown")
checkpoint_path = str(MRI_MODEL_PATH)

print("MODEL VERIFICATION")
print(f"selected model kind: {model_kind}")
print(f"source: {source}")
print(f"checkpoint path: {checkpoint_path}")

MODEL VERIFICATION
selected model kind: resnet50_quantum_attention
source: report:neuroscan_report.json
checkpoint path: c:\Users\saisa\OneDrive\Desktop\Early_Alzheimers\saved_models\best_mri_model.pth


In [10]:
# Section 7B: MRI Dashboard Launch

from pathlib import Path
import os

_required = ["run_analysis_dashboard", "NEUROSCAN_CSS", "MRI_CLASSES"]
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(
        f"Backend globals missing: {_missing}. "
        "Run the Section 7A and 7A+ cells above before launching the UI.",
    )

with gr.Blocks(css=NEUROSCAN_CSS, theme=gr.themes.Base()) as neuroscan_app:

    with gr.Column(elem_id="ns-shell"):

        gr.HTML("""
            <div class='ns-hdr'>
              <div class='ns-hdr-logo'>
                <span class='ns-hdr-badge'>RESEARCH</span>
                <span class='ns-hdr-title'>NeuroScan MRI</span>
              </div>
              <div class='ns-hdr-sub'>
                MRI-only Alzheimer's risk screening and probability analysis
              </div>
              <div class='ns-disclaimer'>
                This tool is a <strong>research screening aid only</strong> and does
                <strong>not</strong> constitute a clinical diagnosis. All results must be
                interpreted by a qualified neurologist.
              </div>
            </div>
        """)

        with gr.Row():
            mri_ui = gr.Image(label="MRI Scan - Required", type="filepath", sources=["upload"], height=290)
            gauge_ui = gr.Plot(label="MRI Risk Gauge", min_width=280)

        with gr.Row():
            p_name = gr.Textbox(label="Patient Name", placeholder="Optional - used in PDF report")
            p_age = gr.Textbox(label="Patient Age", placeholder="Optional")
            tta_ui = gr.Checkbox(value=True, label="Enable Test-Time Augmentation")

        notes_ui = gr.Textbox(
            label="Clinical Notes (optional - included in PDF)",
            lines=2,
            placeholder="Enter any relevant clinical context here",
        )

        run_btn = gr.Button("Run MRI Risk Analysis", variant="primary", size="lg")

        badge_ui = gr.HTML(
            "<div class='card' style='min-height:190px;color:#64748b;display:flex;align-items:center'>"
            "Awaiting MRI scan upload...</div>",
            label="Primary Risk Signal",
        )

        with gr.Row():
            summary_ui = gr.HTML("<div class='card' style='color:#64748b'>Run analysis to see summary.</div>", label="MRI Summary")
            mri_html_ui = gr.HTML("<div class='card' style='color:#64748b'>MRI analysis will appear here.</div>", label="MRI Details")

        class_explain_ui = gr.HTML(
            "<div class='card' style='color:#64748b'>Per-class explanation will appear here.</div>",
            label="Per-Class Explanation",
        )

        with gr.Row():
            radar_ui = gr.Plot(label="MRI Stage Probability Radar")
            gradcam_ui = gr.Image(label="Grad-CAM View (Original | Heatmap | Overlay)", type="numpy", height=360)

        report_json_ui = gr.JSON(label="Structured JSON Report")
        explain_ui = gr.Textbox(label="Decision Explanation (clinical rationale)", lines=14)
        pdf_file_ui = gr.File(label="Download Full Clinical PDF Report")

    run_btn.click(
        fn=run_analysis_dashboard,
        inputs=[mri_ui, tta_ui, p_name, p_age, notes_ui],
        outputs=[
            badge_ui,
            summary_ui,
            mri_html_ui,
            class_explain_ui,
            explain_ui,
            report_json_ui,
            radar_ui,
            gauge_ui,
            gradcam_ui,
            pdf_file_ui,
        ],
        show_progress="full",
    )

AUTO_LAUNCH = os.environ.get("NS_LAUNCH_DASHBOARD", "0") == "1"
if AUTO_LAUNCH:
    print("Launching MRI dashboard...")
    neuroscan_app.queue()
    neuroscan_app.launch(share=False, inline=False)
else:
    print("Dashboard built. To launch:")
    print("  Option 1: Set os.environ['NS_LAUNCH_DASHBOARD'] = '1' then re-run this cell.")
    print("  Option 2: Call neuroscan_app.launch() in a new cell.")

Launching MRI dashboard...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
